# Lab 2b · Quem vai cancelar
**Oficina do Builder · Winovoito** — Bancada 2 · ML aplicado

A **Fluxo**, empresa fictícia de software por assinatura, perde clientes todo mês e quer agir antes. **Guarde bem este modelo:** ele vira a API do Lab 2d e a ferramenta do agente na Bancada 5 e no projeto final.

Neste lab você:
1. encontra uma coluna com vazamento escondida nos dados;
2. vê por que a acurácia engana;
3. treina e compara regressão logística e gradient boosting;
4. calibra as probabilidades;
5. escolhe o limiar pelo **custo**, não pelo 0,5;
6. explica o modelo no geral e para um cliente;
7. salva o modelo com metadados e hash.

In [ ]:
# Setup: baixa o material aberto da Oficina (rode uma vez)
REPO_URL = "https://github.com/winovoito/oficina-do-builder-aberto"  # endereço definitivo a confirmar
import os, sys
if not os.path.exists("oficina-do-builder-aberto"):
    !git clone -q {REPO_URL}
sys.path.insert(0, "oficina-do-builder-aberto")
%pip install -q -r oficina-do-builder-aberto/requirements.txt
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.max_columns", 30)

## 1 · Os dados

In [ ]:
from oficina.sinteticos import gerar_fluxo
from oficina import churn

df = gerar_fluxo(n=20_000, semente=11)
print(df.shape, f"· churn: {df['cancelou'].mean():.1%}")
df.head()

## 2 · Exercício 1 · Ache o vazamento
Uma das colunas foi medida **depois** do momento da previsão e entrega a resposta. Para cada coluna, pergunte: *no instante em que vou prever, eu já sei este valor?*

Uma pista técnica: uma coluna que, sozinha, prevê o alvo com AUC acima de 0,95 é suspeita.

In [ ]:
from sklearn.metrics import roc_auc_score

for c in df.select_dtypes("number").columns.drop(["cancelou", "cliente_id", "mes_referencia"]):
    auc = roc_auc_score(df["cancelou"], df[c].fillna(df[c].median()))
    print(f"{c:32s} AUC sozinha = {max(auc, 1 - auc):.3f}")

# TODO: escreva o nome da coluna com vazamento
minha_resposta = "dias_ultimo_acesso_pos_corte"
print("\n✅ Correto" if [minha_resposta] == churn.detectar_vazamento(df) else "\n❌ Tente de novo")
df = df.drop(columns=[minha_resposta])

## 3 · Dividir no tempo
Treino com os meses 1 a 9, teste com 10 a 12.

In [ ]:
treino, teste = churn.dividir(df)
X = churn.CARACTERISTICAS
Xtr, ytr, Xte, yte = treino[X], treino["cancelou"], teste[X], teste["cancelou"]
print(len(treino), "treino ·", len(teste), "teste")

## 4 · O baseline "ninguém cancela"
Acurácia alta, recall zero. Por isso acurácia não serve para churn.

In [ ]:
from sklearn.metrics import accuracy_score, recall_score
nunca = np.zeros(len(yte))
print(f"Acurácia: {accuracy_score(yte, nunca):.1%} · Recall: {recall_score(yte, nunca):.0%}")

## 5 · Dois modelos, uma comparação
Os dois dentro de pipelines: o preenchimento de faltantes e a escala aprendem só com o treino.

In [ ]:
from sklearn.metrics import RocCurveDisplay

log = churn.pipeline_logistica().fit(Xtr, ytr)
gb = churn.pipeline_boosting().fit(Xtr, ytr)
auc_log = roc_auc_score(yte, log.predict_proba(Xte)[:, 1])
auc_gb = roc_auc_score(yte, gb.predict_proba(Xte)[:, 1])
print(f"AUC regressão logística: {auc_log:.3f} · gradient boosting: {auc_gb:.3f}")

fig, ax = plt.subplots(figsize=(5, 5))
RocCurveDisplay.from_estimator(log, Xte, yte, ax=ax, name="Logística")
RocCurveDisplay.from_estimator(gb, Xte, yte, ax=ax, name="Boosting");

**Pergunta:** a diferença justifica o modelo mais complexo? A regra da Oficina: se o ganho for menor que 0,01 de AUC, fique com o simples e explicável.

In [ ]:
simples_basta = auc_gb - auc_log < 0.01
escolhido = churn.pipeline_logistica() if simples_basta else churn.pipeline_boosting()
print("Escolhido:", "regressão logística" if simples_basta else "gradient boosting")

## 6 · Calibração
O limiar só faz sentido se "0,3" quiser dizer "30% de chance". A calibração ajusta isso.

In [ ]:
from sklearn.calibration import CalibrationDisplay
modelo = churn.calibrar(escolhido, Xtr, ytr)
prob = modelo.predict_proba(Xte)[:, 1]
CalibrationDisplay.from_predictions(yte, prob, n_bins=10, name="calibrado");

## 7 · O limiar é decisão de negócio
Custos fictícios: contato **R$ 15**, cliente perdido **R$ 900**, e o contato segura **25%** de quem ia cancelar.

In [ ]:
grade = np.arange(0.01, 0.96, 0.01)
custos = [churn.custo_total(yte, prob, l) for l in grade]
limiar, custo = churn.escolher_limiar(yte, prob)
custo_05 = churn.custo_total(yte, prob, 0.5)
plt.plot(grade, custos); plt.axvline(limiar, ls="--"); plt.xlabel("limiar"); plt.ylabel("custo (R$)")
print(f"Limiar por custo: {limiar:.2f} → R$ {custo:,.0f} · Limiar 0,5 → R$ {custo_05:,.0f}")
print("Critério do lab:", "✅" if custo < custo_05 else "❌")

## 8 · O que mais pesa, no geral
Importância por permutação: quanto o modelo piora quando você embaralha uma coluna.

> **Importância não é causa.** Se "chamados" pesa muito, pode ser que clientes insatisfeitos abram mais chamados, e não que os chamados causem o cancelamento.

In [ ]:
from sklearn.inspection import permutation_importance
imp = permutation_importance(modelo, Xte, yte, scoring="roc_auc", n_repeats=5, random_state=0)
pd.Series(imp.importances_mean, index=X).sort_values().plot(kind="barh", title="Queda de AUC ao embaralhar");

## 9 · Por que ESTE cliente?
Explicação local: para cada característica, troca o valor do cliente pelo valor típico e mede quanto o risco cai. Esses fatores são o que a API devolve e o que o agente vai usar para justificar uma ação.

> Bônus opcional: compare com valores SHAP (`pip install shap`), biblioteca open source.

In [ ]:
cliente = teste.assign(prob=prob).sort_values("prob").iloc[[-1]]
display(cliente[X + ["prob"]])
for f in churn.fatores_locais(modelo, cliente, treino):
    print(f"• {f['nome']}: {f['valor']} (típico {f['tipico']}) → +{f['efeito']:.0%} de risco")

## 10 · Salvar com metadados
Versão, data, métricas, limiar, características e o **hash** do arquivo. O Lab 2d só carrega o modelo se o hash bater.

In [ ]:
from datetime import date
meta = churn.Metadados(
    versao="churn-1.0.0", data_treino=date.today().isoformat(), caracteristicas=X, limiar=limiar,
    modelo_escolhido="regressao_logistica" if simples_basta else "gradient_boosting",
    metricas={"auc_logistica": round(auc_log, 4), "auc_boosting": round(auc_gb, 4),
              "custo_limiar_escolhido": custo, "custo_limiar_0_5": custo_05},
    observacoes=["Dados sintéticos da Fluxo (empresa fictícia)."])
churn.salvar(modelo, meta, "modelos/churn")
print(open("modelos/churn/metadados.json", encoding="utf-8").read())

## 11 · Bônus opcional · Uma rede neural pequena
Compare uma rede neural com o seu modelo: AUC, tempo de treino e esforço de ajuste. Em dados tabulares deste tamanho, o resultado costuma empatar ou perder.

In [ ]:
import time
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
t0 = time.time()
rede = Pipeline([("pre", churn._preprocessador(True)),
                 ("modelo", MLPClassifier(hidden_layer_sizes=(32, 16), max_iter=300,
                                          early_stopping=True, random_state=0))]).fit(Xtr, ytr)
print(f"Rede neural: AUC {roc_auc_score(yte, rede.predict_proba(Xte)[:, 1]):.3f} em {time.time() - t0:.1f}s")
# Para ir além, refaça com PyTorch ou Keras e anote o esforço.

## Reflexão
1. Que coluna real da sua empresa poderia esconder vazamento parecido?
2. Quem decide os custos da tabela? O que muda no limiar se o contato custar R$ 60?
3. O modelo aprende com o passado. Que grupo de clientes pode estar mal representado?